# 04 - Stage 1 Baseline Classifier

This notebook trains a directly explainable message-stage baseline from existing 200/600/900 codes. It combines word and character TF-IDF features with class-weighted logistic regression.

Validation uses conversation IDs as groups. Messages from one chat are never split across training and test folds. The notebook does not hand-label data or execute training automatically.

In [1]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import FeatureUnion, Pipeline

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

WORKSPACE = Path.cwd()
PROCESSED_DIR = WORKSPACE / "data" / "processed"
MODEL_DIR = WORKSPACE / "models"
REPORT_DIR = WORKSPACE / "reports"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)

MESSAGES_PATH = PROCESSED_DIR / "coded_stage_messages.csv"
if not MESSAGES_PATH.exists():
    raise FileNotFoundError("Run 01_coded_chat_data_loading.ipynb first.")

messages = pd.read_csv(MESSAGES_PATH, dtype={"category": "string"})
messages["text"] = messages["text"].fillna("")
messages["category"] = messages["category"].astype(str)

VALID_CATEGORIES = {"200", "600", "900"}
assert messages["category"].isin(VALID_CATEGORIES).all()
assert not messages.duplicated(["chat_id", "msg_idx"]).any()
assert messages["chat_id"].notna().all()

print("Messages:", len(messages))
print("Chats:", messages["chat_id"].nunique())
print(messages["category"].value_counts().sort_index())

Messages: 10831
Chats: 56
category
200    1896
600    4715
900    4220
Name: count, dtype: int64


## Context representation

For each target message, the model receives the previous message and next message as separated context. Context is included only when available; no neutral or synthetic messages are invented.

In [2]:
def build_context_text(frame: pd.DataFrame) -> pd.DataFrame:
    context_frame = frame.sort_values(["chat_id", "msg_idx"]).copy()
    context_frame["previous_text"] = context_frame.groupby("chat_id", sort=False)["text"].shift(1).fillna("")
    context_frame["next_text"] = context_frame.groupby("chat_id", sort=False)["text"].shift(-1).fillna("")
    context_frame["model_text"] = (
        "previous: " + context_frame["previous_text"] + " target: " + context_frame["text"] + " next: " + context_frame["next_text"]
    )
    return context_frame


model_data = build_context_text(messages)
display(model_data[["chat_id", "msg_idx", "category", "model_text"]].head())

,chat_id,msg_idx,category,model_text
0,ArmySgt1961,1,200,previous: target: im dennis us army soldier f...
1,ArmySgt1961,3,200,previous: im dennis us army soldier from cinci...
2,ArmySgt1961,5,200,previous: how old ru target: u single next: ok...
3,ArmySgt1961,8,600,previous: u single target: ok u have sex at 13...
4,ArmySgt1961,18,200,previous: ok u have sex at 13 target: u have a...


## Grouped repeated validation

A small 56-chat sample requires repeated grouped cross-validation and confidence intervals. `StratifiedGroupKFold` preserves approximate class balance while keeping every conversation intact. Do not use ordinary `train_test_split` or `KFold` on this data.

In [4]:
def build_baseline_pipeline() -> Pipeline:
    features = FeatureUnion([
        ("word_tfidf", TfidfVectorizer(
            analyzer="word", ngram_range=(1, 3), min_df=2,
            sublinear_tf=True, lowercase=True
        )),
        ("character_tfidf", TfidfVectorizer(
            analyzer="char_wb", ngram_range=(2, 5), min_df=2,
            sublinear_tf=True, lowercase=True
        )),
    ])
    classifier = LogisticRegression(
        class_weight="balanced",
        max_iter=2000,
        solver="lbfgs",
        random_state=RANDOM_SEED,
    )
    return Pipeline([
        ("tfidf", features),
        ("classifier", classifier),
    ])


X = model_data["model_text"].to_numpy()
y = model_data["category"].to_numpy()
groups = model_data["chat_id"].to_numpy()

n_splits = 5 if model_data["chat_id"].nunique() >= 5 else model_data["chat_id"].nunique()
n_repeats = 3
fold_rows = []
prediction_rows = []

for repeat in range(n_repeats):
    splitter = StratifiedGroupKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_SEED + repeat,
    )
    for fold, (train_index, test_index) in enumerate(splitter.split(X, y, groups)):
        pipeline = build_baseline_pipeline()
        pipeline.fit(X[train_index], y[train_index])
        predictions = pipeline.predict(X[test_index])

        report = classification_report(
            y[test_index], predictions,
            labels=["200", "600", "900"],
            output_dict=True,
            zero_division=0,
        )
        macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(
            y[test_index], predictions,
            labels=["200", "600", "900"],
            average="macro",
            zero_division=0,
        )
        fold_rows.append({
            "repeat": repeat,
            "fold": fold,
            "train_messages": len(train_index),
            "test_messages": len(test_index),
            "macro_precision": macro_precision,
            "macro_recall": macro_recall,
            "macro_f1": macro_f1,
            "f1_200": report["200"]["f1-score"],
            "f1_600": report["600"]["f1-score"],
            "f1_900": report["900"]["f1-score"],
        })

        fold_predictions = pd.DataFrame({
            "repeat": repeat,
            "fold": fold,
            "chat_id": groups[test_index],
            "msg_idx": model_data.iloc[test_index]["msg_idx"].to_numpy(),
            "true_category": y[test_index],
            "predicted_category": predictions,
        })
        prediction_rows.append(fold_predictions)

fold_metrics = pd.DataFrame(fold_rows)
fold_predictions = pd.concat(prediction_rows, ignore_index=True)

metric_summary = fold_metrics[[
    "macro_precision", "macro_recall", "macro_f1", "f1_200", "f1_600", "f1_900"
]].agg(["mean", "std"]).T
metric_summary.columns = ["mean", "std"]

display(fold_metrics)
display(metric_summary)


,repeat,fold,train_messages,test_messages,macro_precision,macro_recall,macro_f1,f1_200,f1_600,f1_900
0,0,0,8719,2112,0.649840,0.657524,0.647942,0.541860,0.725203,0.676761
1,0,1,8727,2104,0.718501,0.696624,0.704893,0.591750,0.779894,0.743034
2,0,2,8642,2189,0.732212,0.738773,0.732304,0.646831,0.773230,0.776850
3,0,3,8700,2131,0.724413,0.695299,0.704985,0.594675,0.777358,0.742923
4,0,4,8536,2295,0.678275,0.679753,0.675762,0.594164,0.743756,0.689367
5,1,0,8719,2112,0.649840,0.657524,0.647942,0.541860,0.725203,0.676761
6,1,1,8727,2104,0.718501,0.696624,0.704893,0.591750,0.779894,0.743034
7,1,2,8642,2189,0.732212,0.738773,0.732304,0.646831,0.773230,0.776850
8,1,3,8700,2131,0.724413,0.695299,0.704985,0.594675,0.777358,0.742923
9,1,4,8536,2295,0.678275,0.679753,0.675762,0.594164,0.743756,0.689367


,mean,std
macro_precision,0.700648,0.032607
macro_recall,0.693595,0.027565
macro_f1,0.693177,0.029846
f1_200,0.593856,0.034377
f1_600,0.759888,0.022425
f1_900,0.725787,0.038532


## Final model and artifacts

The final model is fit on all coded predator messages only after grouped cross-validation. Saved predictions and metrics should be interpreted with the small-sample uncertainty reported above.

In [5]:
final_pipeline = build_baseline_pipeline()
final_pipeline.fit(X, y)

joblib.dump(final_pipeline, MODEL_DIR / "stage1_tfidf_logistic.joblib")
model_data.to_csv(PROCESSED_DIR / "stage1_model_input.csv", index=False)
fold_metrics.to_csv(REPORT_DIR / "stage1_fold_metrics.csv", index=False)
fold_predictions.to_csv(REPORT_DIR / "stage1_fold_predictions.csv", index=False)
metric_summary.to_csv(REPORT_DIR / "stage1_metric_summary.csv")

with (REPORT_DIR / "stage1_validation_config.json").open("w", encoding="utf-8") as handle:
    json.dump({
        "random_seed": RANDOM_SEED,
        "group_column": "chat_id",
        "splitter": "StratifiedGroupKFold",
        "n_splits": n_splits,
        "n_repeats": n_repeats,
        "word_ngrams": [1, 3],
        "character_ngrams": [2, 5],
        "classes": sorted(VALID_CATEGORIES),
    }, handle, indent=2)

print("Saved Stage 1 baseline and validation artifacts.")

Saved Stage 1 baseline and validation artifacts.


## Limitations

* All training conversations are known predatory; this model predicts a stage, not whether a conversation is benign or predatory.
* Coded classes can be rare, especially 900, so per-class confidence intervals should accompany this baseline.
* Grouped validation prevents message leakage but cannot eliminate dataset-style or source artifacts.
* The later lexical and embedding models should be compared against this fixed, reproducible baseline.